# M3 step 3: JPEG and resize fine-tuning

The clean U-Net from step 2 recovered messages reliably from PNG images, but its JPEG results were close to guessing. This notebook fine-tunes that model with simulated JPEG, real JPEG, resizing and an identity branch.

The run starts from the clean epoch-60 weights. The selected checkpoint is the final model from these experiments; the summary below records both its improvements and its remaining limits.

## How the distortion training works

A distortion is applied between the encoder and decoder so the networks can adapt to the images the decoder will receive after editing or compression.

First, the stamped image is clipped and rounded to 8-bit pixel values. A straight-through operation passes gradients through this rounding step. One branch is then chosen for the whole batch, with a probability of one quarter for each:

1. **JPEG-Mask:** the differentiable approximation from the earlier notebooks. Message-loss gradients can reach both encoder and decoder.
2. **Real JPEG at quality 50:** Pillow encodes and decodes the image using 4:2:0 chroma subsampling. The message loss updates the decoder but has no gradient path to the encoder. The encoder still receives its image and adversarial losses.
3. **Resize:** a random scale factor from 0.5 to 2.0 is applied, then the image is resized back to 128×128. Training uses bilinear interpolation with antialiasing.
4. **Identity:** no further distortion is added after the 8-bit conversion.

The real/simulated JPEG mixture is adapted from MBRS. This run adds resizing and retains the earlier HiDDeN loss weights and JPEG-Mask implementation. The learning rate is reduced from 0.001 to 0.0001 for fine-tuning.

## Training setup and time

The run used PyTorch 2.4.1 on an NVIDIA GeForce RTX 2070 Super with Max-Q Design. The 100 logged epochs took about 8 hours 6 minutes, excluding testing.

## 1. Settings

The image, message and adversarial loss weights remain 0.7, 1.0 and 0.001. Adam uses a learning rate of 0.0001. `MIN_PSNR` sets the 30 dB threshold used by the checkpoint selection score.

In [ ]:
import os, io, csv, json, time, random, shutil, hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

ENCODER = 'unet'
NOISE = 'jpegmask+realjpeg50+resize+identity'
RUNS = Path(r'E:\VeriStamp_M3\hidden_runs')
INIT_FROM = RUNS / 'hidden_unet_none_10k' / 'latest.pt'      # trained U-Net from step 2
RUN_NAME = 'hidden_unet_jpegresize_v2_10k'

SEED = 42
N_TRAIN = 10000
IMAGE_SIZE = 128
MESSAGE_BITS = 32
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 1e-4
IMAGE_LOSS, MESSAGE_LOSS, ADVERSARIAL_LOSS = 0.7, 1.0, 1e-3
TRAIN_JPEG_QUALITY = 50
RESIZE_RANGE = (0.5, 2.0)
MIN_PSNR = 30.0
NUM_WORKERS = 0
RESUME = True

DATA = Path(r'E:\VeriStamp_M3\hidden_data')
RUN_DIR = RUNS / RUN_NAME

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Switch the kernel to "Deep Learning".')
if not INIT_FROM.exists():
    raise FileNotFoundError(f'Clean U-Net weights not found: {INIT_FROM}')
device = torch.device('cuda')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG = RUN_DIR / 'train_log.txt'
def log(message):
    print(message, flush=True)
    with open(LOG, 'a', encoding='utf-8') as f:
        f.write(message + '\n')

config = {'encoder': ENCODER, 'noise': NOISE, 'init_from': str(INIT_FROM), 'seed': SEED, 'n_train': N_TRAIN,
          'image_size': IMAGE_SIZE, 'message_bits': MESSAGE_BITS, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
          'learning_rate': LEARNING_RATE, 'image_loss': IMAGE_LOSS, 'message_loss': MESSAGE_LOSS,
          'adversarial_loss': ADVERSARIAL_LOSS, 'train_jpeg_quality': TRAIN_JPEG_QUALITY,
          'resize_range': list(RESIZE_RANGE), 'min_psnr': MIN_PSNR, 'jpeg_subsampling': '4:2:0'}
print('PyTorch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
print('Starting weights:', INIT_FROM)
print('Run folder:', RUN_DIR)

## 2. Data

This run reuses the 10,000 training images and the 500-image validation and test splits from the clean runs. No new images are downloaded here.

In [ ]:
folders = {'train': DATA / 'train', 'validation': DATA / 'val', 'test': DATA / 'test'}
counts = {name: len(list(folder.glob('*.jpg'))) for name, folder in folders.items()}
print('Image counts:', counts)
assert counts == {'train': N_TRAIN, 'validation': 500, 'test': 500}, \
    'Data folder incomplete. It is created by the conv/U-Net clean notebooks (section 2).'

## 3. Payload

The payload has 32 bits: 2 for the verdict, 4 for the AI-probability bucket, 4 for the detector ID, 14 for an image tag and 8 for a CRC checksum. These experiments use repeatable sample payloads generated from each filename, rather than predictions from M1, M2 or the fusion model.

The CRC checks for transmission errors, but a matching checksum does not guarantee that every bit is correct. The results therefore report bit accuracy, exact 32-bit recovery and CRC acceptance separately.

Validation and test images receive fixed sample payloads. Training continues to use new random 32-bit messages in each batch.

In [ ]:
VERDICT_CODES = {'Real': 0, 'AI': 1, 'Uncertain': 2}
CODE_VERDICTS = {v: k for k, v in VERDICT_CODES.items()}

def int_to_bits(value, width):
    return [(value >> s) & 1 for s in range(width - 1, -1, -1)]

def bits_to_int(bits):
    out = 0
    for b in bits:
        out = (out << 1) | int(b)
    return out

def crc8(bits):
    reg = 0
    for b in bits:
        fb = ((reg >> 7) & 1) ^ int(b)
        reg = (reg << 1) & 255
        if fb:
            reg ^= 0x07
    return int_to_bits(reg, 8)

def encode_payload(verdict, p_ai, detector_id=3, image_tag=0):
    fields = (int_to_bits(VERDICT_CODES[verdict], 2) + int_to_bits(round(max(0.0, min(1.0, p_ai)) * 15), 4)
              + int_to_bits(detector_id, 4) + int_to_bits(image_tag & 0x3FFF, 14))
    return fields + crc8(fields)

def decode_payload(bits):
    code = bits_to_int(bits[:2])
    valid = code in CODE_VERDICTS and crc8(bits[:24]) == list(bits[24:])
    return {'valid': valid, 'verdict': CODE_VERDICTS[code] if valid else None}

def fixed_payload(filename):
    digest = hashlib.sha256(f'{SEED}:payload:{filename}'.encode()).digest()
    rng = random.Random(int.from_bytes(digest[:8], 'big'))
    p_ai = rng.random()
    verdict = 'Real' if p_ai < 0.35 else ('AI' if p_ai > 0.65 else 'Uncertain')
    return encode_payload(verdict, p_ai, 3, rng.randrange(2**14))

example = encode_payload('Real', 0.1, image_tag=123)
assert decode_payload(example)['verdict'] == 'Real'
assert all(not decode_payload(example[:i] + [1 - example[i]] + example[i + 1:])['valid'] for i in range(32))
print('Payload functions OK')

## 4. Loading the photos

Training uses random 128×128 crops. Validation and testing both resize the shorter side to 128 pixels and take a centre crop. They therefore use the same preprocessing, unlike the validation and test paths in the clean notebooks.

In [ ]:
class Photos(Dataset):
    def __init__(self, folder, train):
        self.files = sorted(Path(folder).glob('*.jpg'))
        if train:
            first = [transforms.RandomCrop(IMAGE_SIZE, pad_if_needed=True)]
        else:
            first = [transforms.Resize(IMAGE_SIZE), transforms.CenterCrop(IMAGE_SIZE)]
        self.transform = transforms.Compose(first + [transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        with Image.open(self.files[index]) as image:
            return self.transform(image.convert('RGB')), index

train_set, val_set = Photos(folders['train'], True), Photos(folders['validation'], False)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_messages = torch.tensor([fixed_payload(p.name) for p in val_set.files], dtype=torch.float32)
print('Train', len(train_set), '| validation', len(val_set), '| steps per epoch', len(train_loader))

## 5. Model

The U-Net encoder, HiDDeN-style decoder and discriminator are unchanged from step 2.

In [ ]:
class ConvBNRelu(nn.Module):
    def __init__(self, channels_in, channels_out, stride=1):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(channels_in, channels_out, 3, stride, padding=1),
                                    nn.BatchNorm2d(channels_out), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.layers(x)


class Decoder(nn.Module):
    def __init__(self, channels=64, blocks=7, message_bits=32):
        super().__init__()
        layers = [ConvBNRelu(3, channels)] + [ConvBNRelu(channels, channels) for _ in range(blocks - 1)]
        layers += [ConvBNRelu(channels, message_bits), nn.AdaptiveAvgPool2d(1)]
        self.layers = nn.Sequential(*layers)
        self.linear = nn.Linear(message_bits, message_bits)

    def forward(self, image):
        return self.linear(self.layers(image).flatten(1))


class Discriminator(nn.Module):
    def __init__(self, channels=64, blocks=3):
        super().__init__()
        layers = [ConvBNRelu(3, channels)] + [ConvBNRelu(channels, channels) for _ in range(blocks - 1)]
        self.before_linear = nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1))
        self.linear = nn.Linear(channels, 1)

    def forward(self, image):
        return self.linear(self.before_linear(image).flatten(1))


class UNetEncoder(nn.Module):
    def __init__(self, channels=64, message_bits=32):
        super().__init__()
        c, m = channels, message_bits
        self.inc = nn.Sequential(ConvBNRelu(3 + m, c), ConvBNRelu(c, c))
        self.down1 = nn.Sequential(ConvBNRelu(c, 2 * c, stride=2), ConvBNRelu(2 * c, 2 * c))
        self.down2 = nn.Sequential(ConvBNRelu(2 * c, 4 * c, stride=2), ConvBNRelu(4 * c, 4 * c))
        self.bottleneck = ConvBNRelu(4 * c + m, 4 * c)
        self.up2 = nn.Sequential(ConvBNRelu(4 * c + 2 * c, 2 * c), ConvBNRelu(2 * c, 2 * c))
        self.up1 = nn.Sequential(ConvBNRelu(2 * c + c, c), ConvBNRelu(c, c))
        self.after_concat_layer = ConvBNRelu(c + 3 + m, c)
        self.final_layer = nn.Conv2d(c, 3, kernel_size=1)

    @staticmethod
    def _expand(message, like):
        return message.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, like.shape[-2], like.shape[-1])

    def forward(self, image, message):
        x0 = self.inc(torch.cat([image, self._expand(message, image)], dim=1))
        x1 = self.down1(x0)
        x2 = self.down2(x1)
        x2 = self.bottleneck(torch.cat([x2, self._expand(message, x2)], dim=1))
        u2 = self.up2(torch.cat([F.interpolate(x2, size=x1.shape[-2:], mode='nearest'), x1], dim=1))
        u1 = self.up1(torch.cat([F.interpolate(u2, size=x0.shape[-2:], mode='nearest'), x0], dim=1))
        return self.final_layer(self.after_concat_layer(torch.cat([self._expand(message, image), u1, image], dim=1)))

## 6. Distortions

`apply_noise` selects the training branch described above. Simulated JPEG and training-time resizing use tensor operations; the real-JPEG branch uses Pillow.

For validation and testing, `pil_apply` converts each image to 8-bit RGB pixels and applies the requested Pillow operations. JPEG conditions include actual JPEG encoding and decoding in memory. The condition named `png_saved` uses the resulting 8-bit pixels without a separate PNG encode/decode; it is a PNG-equivalent pixel conversion, not a PNG file-writing test. Resize and crop conditions also operate in memory.

In [ ]:
BILINEAR = getattr(Image, 'Resampling', Image).BILINEAR

def pil_apply(images01, ops):
    """Apply real edits such as [('jpeg', 70), ('resize', 224)] to a batch with values from 0 to 1."""
    out = []
    for image in images01.detach().clamp(0, 1).cpu():
        pil = Image.fromarray((image.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))
        for op, value in ops:
            if op == 'jpeg':
                buffer = io.BytesIO()
                pil.save(buffer, format='JPEG', quality=value, subsampling=2)
                buffer.seek(0)
                with Image.open(buffer) as restored:
                    pil = restored.convert('RGB')
            elif op == 'resize':
                pil = pil.resize((value, value), BILINEAR)
            elif op == 'crop':
                left = (pil.width - value) // 2
                top = (pil.height - value) // 2
                pil = pil.crop((left, top, left + value, top + value))
            else:
                raise ValueError(op)
        out.append(torch.from_numpy(np.asarray(pil, dtype=np.float32) / 255).permute(2, 0, 1))
    return torch.stack(out).to(images01.device)


def _dct_coeff(n, k, N):
    return np.cos(np.pi / N * (n + 0.5) * k)

def _idct_coeff(n, k, N):
    return (int(n == 0) * (-0.5) + np.cos(np.pi / N * (k + 0.5) * n)) * np.sqrt(1 / (2.0 * N))

def _filters(fn):
    filters = np.zeros((64, 8, 8))
    for ky in range(8):
        for kx in range(8):
            for ny in range(8):
                for nx in range(8):
                    filters[ky * 8 + kx, ny, nx] = fn(ny, ky, 8) * fn(nx, kx, 8)
    return torch.tensor(filters, dtype=torch.float32).unsqueeze(1)

def _zigzag_mask(keep):
    order = sorted(((x, y) for x in range(8) for y in range(8)),
                   key=lambda p: (p[0] + p[1], -p[1] if (p[0] + p[1]) % 2 else p[1]))
    mask = np.zeros((8, 8), dtype=np.float32)
    for i, j in order[:keep]:
        mask[i, j] = 1
    return mask

class JpegCompression(nn.Module):
    def __init__(self, size=128, keep=(25, 9, 9)):
        super().__init__()
        self.register_buffer('dct', _filters(_dct_coeff))
        self.register_buffer('idct', _filters(_idct_coeff))
        self.register_buffer('mask', torch.tensor(np.stack([np.tile(_zigzag_mask(k), (size // 8, size // 8)) for k in keep])))

    @staticmethod
    def _blockwise(image, filters):
        out = []
        for c in range(image.shape[1]):
            x = F.conv2d(image[:, c:c + 1], filters, stride=8)
            b, _, h8, w8 = x.shape
            x = x.permute(0, 2, 3, 1).reshape(b, h8, w8, 8, 8).permute(0, 1, 3, 2, 4).reshape(b, h8 * 8, w8 * 8)
            out.append(x.unsqueeze(1))
        return torch.cat(out, dim=1)

    def forward(self, x):
        r, g, b = x[:, 0], x[:, 1], x[:, 2]
        yuv = torch.stack([0.299 * r + 0.587 * g + 0.114 * b,
                           -0.14713 * r - 0.28886 * g + 0.436 * b,
                           0.615 * r - 0.51499 * g - 0.10001 * b], dim=1)
        y = self._blockwise(self._blockwise(yuv, self.dct) * self.mask, self.idct)
        return torch.stack([y[:, 0] + 1.13983 * y[:, 2],
                            y[:, 0] - 0.39465 * y[:, 1] - 0.58060 * y[:, 2],
                            y[:, 0] + 2.03211 * y[:, 1]], dim=1)

jpeg_mask = JpegCompression(IMAGE_SIZE).to(device)
BRANCHES = ['jpeg_mask', 'jpeg_real', 'resize', 'identity']

def as_saved_file(x):
    """Round to 8-bit pixel values but keep the gradient."""
    q = torch.round((x.clamp(-1, 1) + 1) * 127.5) / 127.5 - 1
    return x + (q - x).detach()

def resize_noise(x):
    size = max(16, int(round(IMAGE_SIZE * random.uniform(*RESIZE_RANGE))))
    y = F.interpolate(x, size=(size, size), mode='bilinear', align_corners=False, antialias=True)
    return F.interpolate(y, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False, antialias=True)

def apply_noise(encoded):
    x = as_saved_file(encoded)
    branch = random.choice(BRANCHES)
    if branch == 'jpeg_mask':
        return jpeg_mask(x), branch
    if branch == 'jpeg_real':
        with torch.no_grad():
            return pil_apply((x + 1) / 2, [('jpeg', TRAIN_JPEG_QUALITY)]) * 2 - 1, branch
    if branch == 'resize':
        return resize_noise(x), branch
    return x, branch
print('Distortions ready:', BRANCHES)

## 7. Prepare the networks and starting weights

Create the same three networks and fresh Adam optimizers. Read the clean U-Net checkpoint for initialization in the training cell, then check the distortion output shapes.

In [ ]:
encoder = UNetEncoder(message_bits=MESSAGE_BITS).to(device)
decoder = Decoder(message_bits=MESSAGE_BITS).to(device)
discriminator = Discriminator().to(device)
opt_g = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=LEARNING_RATE)
opt_d = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE)
bce, mse = nn.BCEWithLogitsLoss(), nn.MSELoss()

init_state = torch.load(INIT_FROM, map_location=device)
if init_state['config'].get('encoder') != 'unet' or init_state['config'].get('message_bits') != MESSAGE_BITS:
    raise RuntimeError(f'{INIT_FROM} is not the clean 32-bit U-Net run.')
print('Clean U-Net checkpoint: epoch', init_state['epoch'])

with torch.no_grad():
    x = torch.rand(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device) * 2 - 1
    m = torch.randint(0, 2, (2, MESSAGE_BITS), device=device).float()
    stamped = encoder(x, m)
    for branch in BRANCHES:
        random.seed(BRANCHES.index(branch))
        assert apply_noise(stamped)[0].shape == x.shape
random.seed(SEED)
print('Shape checks passed')

## 8. Training

The discriminator and encoder/decoder updates follow the earlier notebooks. The decoder now receives the output of the selected distortion branch:

`loss = 0.7 × image MSE + 1.0 × message MSE + 0.001 × adversarial BCE`

After each epoch, validation measures recovery from 8-bit pixels, real JPEG at quality 50, and a 128→224→128 resize. PSNR compares the 8-bit stamped image with the original.

The checkpoint score ranks epochs below 30 dB behind those meeting the threshold. Among qualifying epochs, it uses JPEG-50 bit accuracy plus a small PSNR term. It selected epoch 97.

During this run, validation PSNR fell to about 28.4 dB before recovering to around 30 dB. At epoch 97, JPEG-50 validation bit accuracy was 86.52% and PSNR was 30.20 dB.

In [ ]:
def train_step(images, messages):
    encoder.train(); decoder.train(); discriminator.train()
    ones = torch.ones(images.shape[0], 1, device=device)
    zeros = torch.zeros(images.shape[0], 1, device=device)

    opt_d.zero_grad(set_to_none=True)
    d_cover = bce(discriminator(images), ones)
    d_cover.backward()
    encoded = encoder(images, messages)
    noised, branch = apply_noise(encoded)
    decoded = decoder(noised)
    d_encoded = bce(discriminator(encoded.detach()), zeros)
    d_encoded.backward()
    opt_d.step()

    opt_g.zero_grad(set_to_none=True)
    g_adv = bce(discriminator(encoded), ones)
    g_enc = mse(encoded, images)
    g_dec = mse(decoded, messages)
    loss = ADVERSARIAL_LOSS * g_adv + IMAGE_LOSS * g_enc + MESSAGE_LOSS * g_dec
    if not torch.isfinite(loss):
        raise RuntimeError('Loss became NaN/inf. Stop and check this run.')
    loss.backward()
    opt_g.step()
    bits = ((decoded.detach() > 0.5).float() == messages).float().mean().item()
    return branch, {'loss': loss.item(), 'image_mse': g_enc.item(), 'message_mse': g_dec.item(), 'bits': bits}

def read_bits(decoder_net, images01):
    return (decoder_net(images01 * 2 - 1) > 0.5).float()

VAL_CONDITIONS = {'png': [], 'jpeg50': [('jpeg', 50)], 'resize224': [('resize', 224), ('resize', IMAGE_SIZE)]}

@torch.no_grad()
def validate():
    encoder.eval(); decoder.eval()
    sums, n = defaultdict(float), 0
    for images, index in val_loader:
        images, messages = images.to(device), val_messages[index].to(device)
        originals01 = (images + 1) / 2
        stamped01 = ((encoder(images, messages) + 1) / 2).clamp(0, 1)
        for name, ops in VAL_CONDITIONS.items():
            correct = read_bits(decoder, pil_apply(stamped01, ops)) == messages
            sums[f'val_{name}_bits'] += correct.float().mean(1).sum().item()
            sums[f'val_{name}_exact'] += correct.all(1).float().sum().item()
        png = pil_apply(stamped01, [])
        sums['val_psnr_db'] += (-10 * torch.log10((png - originals01).square().flatten(1).mean(1).clamp_min(1e-12))).sum().item()
        n += len(images)
    return {k: v / n for k, v in sums.items()}

def selection_score(row):
    """Higher is better: JPEG 50 accuracy, with epochs below MIN_PSNR ranked last."""
    return row['val_jpeg50_bits'] + 1e-4 * row['val_psnr_db'] - (1.0 if row['val_psnr_db'] < MIN_PSNR else 0.0)

def save_checkpoint(path, epoch, history):
    state = {'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(), 'discriminator': discriminator.state_dict(),
             'opt_g': opt_g.state_dict(), 'opt_d': opt_d.state_dict(), 'epoch': epoch, 'history': history, 'config': config}
    torch.save(state, str(path) + '.tmp')
    os.replace(str(path) + '.tmp', path)

def write_history(history):
    with open(RUN_DIR / 'history.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)

def describe(row):
    return (f"val PNG {row['val_png_bits']:.2%} | JPEG50 {row['val_jpeg50_bits']:.2%} (exact {row['val_jpeg50_exact']:.1%}) | "
            f"resize224 {row['val_resize224_bits']:.2%} | PSNR {row['val_psnr_db']:.2f} dB")

(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2))
print('Training helpers ready.')

In [ ]:
history, first_epoch = [], 1
latest = RUN_DIR / 'latest.pt'
same = lambda a, b: {k: v for k, v in a.items() if k != 'epochs'} == {k: v for k, v in b.items() if k != 'epochs'}
if RESUME and latest.exists():
    state = torch.load(latest, map_location=device)
    if not same(state['config'], config):
        raise RuntimeError('This run folder has different settings. Change RUN_NAME or restore the settings.')
    for net, key in [(encoder, 'encoder'), (decoder, 'decoder'), (discriminator, 'discriminator'), (opt_g, 'opt_g'), (opt_d, 'opt_d')]:
        net.load_state_dict(state[key])
    history, first_epoch = state['history'], state['epoch'] + 1
    log(f'Resuming {RUN_NAME} at epoch {first_epoch}')
else:
    encoder.load_state_dict(init_state['encoder'])
    decoder.load_state_dict(init_state['decoder'])
    discriminator.load_state_dict(init_state['discriminator'])
    log(f'Starting {RUN_NAME} from {INIT_FROM} (clean epoch {init_state["epoch"]}), fresh optimizers: {json.dumps(config)}')
    log('Start (clean weights)  | ' + describe(validate()))

best_score = max([selection_score(r) for r in history], default=-float('inf'))
for epoch in range(first_epoch, EPOCHS + 1):
    started = time.time()
    totals, n = defaultdict(float), 0
    branch_bits, branch_n = defaultdict(float), defaultdict(int)
    for images, _ in train_loader:
        images = images.to(device, non_blocking=True)
        messages = torch.randint(0, 2, (images.shape[0], MESSAGE_BITS), device=device).float()
        branch, stats = train_step(images, messages)
        for key, value in stats.items():
            totals[key] += value * len(images)
        branch_bits[branch] += stats['bits'] * len(images)
        branch_n[branch] += len(images)
        n += len(images)
    row = {'epoch': epoch, **{f'train_{k}': v / n for k, v in totals.items()},
           **{f'train_bits_{b}': (branch_bits[b] / branch_n[b]) if branch_n[b] else float('nan') for b in BRANCHES},
           **validate(), 'minutes': (time.time() - started) / 60}
    history.append(row)
    write_history(history)
    improved = selection_score(row) > best_score
    if improved:
        best_score = selection_score(row)
        save_checkpoint(RUN_DIR / 'best.pt', epoch, history)
    save_checkpoint(latest, epoch, history)
    log(f"Epoch {epoch:3d}/{EPOCHS} | train bits {row['train_bits']:.2%} (real JPEG {row['train_bits_jpeg_real']:.2%}) | "
        + describe(row) + f" | {row['minutes']:.2f} min" + (' *best*' if improved else ''))
log('Training finished.')

## 9. Results

### Learning curves

The plots show validation bit accuracy for each condition, validation PSNR and training bit accuracy by distortion branch. PNG-equivalent and resize recovery became much stronger than JPEG-50 recovery. JPEG-50 validation accuracy fluctuated later in training; the last epoch was not the selected checkpoint.

In [ ]:
with open(RUN_DIR / 'history.csv', newline='') as f:
    rows = [{k: float(v) for k, v in r.items()} for r in csv.DictReader(f)]
epochs = [r['epoch'] for r in rows]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for key, label in [('val_png_bits', 'PNG'), ('val_jpeg50_bits', 'Real JPEG Q50'), ('val_resize224_bits', 'Resize 224')]:
    axes[0].plot(epochs, [r[key] * 100 for r in rows], label=label)
axes[0].set(title='Validation bit accuracy (real files)', xlabel='Epoch', ylabel='%', ylim=(40, 101)); axes[0].legend()
axes[1].plot(epochs, [r['val_psnr_db'] for r in rows], color='#437346')
axes[1].axhline(MIN_PSNR, color='grey', linestyle='--', linewidth=1)
axes[1].set(title='Validation PSNR (saved PNG)', xlabel='Epoch', ylabel='dB')
for branch in BRANCHES:
    axes[2].plot(epochs, [r[f'train_bits_{branch}'] * 100 for r in rows], label=branch)
axes[2].set(title='Training bits by distortion', xlabel='Epoch', ylabel='%', ylim=(40, 101)); axes[2].legend()
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
fig.savefig(RUN_DIR / 'learning_curves.png', bbox_inches='tight')
plt.show()

### Test on 500 held-out images

The clean epoch-60 model and the selected fine-tuned model are evaluated with the same images, payloads and preprocessing. This gives the before/after comparison for fine-tuning. The clean baseline here differs from the epoch-48 checkpoint tested in step 2.

The conditions are 8-bit conversion; real JPEG at qualities 90, 70 and 50; 128→224→128 and 128→64→128 resizing; JPEG 70 followed by 128→224→128 resizing; and a 112×112 centre crop. The crop is decoded at 112×112 and was not part of training. Resize tests use Pillow bilinear interpolation.

The 224-pixel result describes the round trip back to 128. It does not measure decoding directly at 224×224 or the effect of stamping on the AI detectors' predictions.

In [ ]:
TEST_CONDITIONS = {
    'png_saved': [],
    'jpeg90': [('jpeg', 90)], 'jpeg70': [('jpeg', 70)], 'jpeg50': [('jpeg', 50)],
    'resize224': [('resize', 224), ('resize', IMAGE_SIZE)],
    'resize64': [('resize', 64), ('resize', IMAGE_SIZE)],
    'jpeg70_then_resize224': [('jpeg', 70), ('resize', 224), ('resize', IMAGE_SIZE)],
    'crop112': [('crop', 112)],
}
test_transform = transforms.Compose([transforms.Resize(IMAGE_SIZE), transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor()])
test_files = sorted(folders['test'].glob('*.jpg'))

def load_model(path):
    state = torch.load(path, map_location=device)
    enc, dec = UNetEncoder(message_bits=MESSAGE_BITS).to(device), Decoder(message_bits=MESSAGE_BITS).to(device)
    enc.load_state_dict(state['encoder']); dec.load_state_dict(state['decoder'])
    return enc.eval(), dec.eval(), state['epoch']

@torch.no_grad()
def run_test(enc, dec, label):
    records, psnr_sum, unstamped_valid = [], 0.0, 0
    for start in range(0, len(test_files), BATCH_SIZE):
        files = test_files[start:start + BATCH_SIZE]
        images01 = []
        for path in files:
            with Image.open(path) as image:
                images01.append(test_transform(image.convert('RGB')))
        images01 = torch.stack(images01).to(device)
        messages = torch.tensor([fixed_payload(p.name) for p in files], dtype=torch.float32, device=device)
        stamped01 = ((enc(images01 * 2 - 1, messages) + 1) / 2).clamp(0, 1)
        png = pil_apply(stamped01, [])
        psnr_sum += (-10 * torch.log10((png - images01).square().flatten(1).mean(1).clamp_min(1e-12))).sum().item()
        unstamped = read_bits(dec, pil_apply(images01, [])).long().cpu().tolist()
        unstamped_valid += sum(decode_payload(b)['valid'] for b in unstamped)
        truth = messages.long().cpu().tolist()
        variants = {'float_clean': stamped01, **{name: pil_apply(stamped01, ops) for name, ops in TEST_CONDITIONS.items()}}
        for condition, images in variants.items():
            for path, expected, actual in zip(files, truth, read_bits(dec, images).long().cpu().tolist()):
                got, want = decode_payload(actual), decode_payload(expected)
                records.append({'model': label, 'filename': path.name, 'condition': condition,
                                'bit_accuracy': sum(a == b for a, b in zip(actual, expected)) / 32,
                                'exact_message': int(actual == expected), 'stamp_valid': int(got['valid']),
                                'valid_correct_verdict': int(got['valid'] and got['verdict'] == want['verdict']),
                                'expected_bits': ''.join(map(str, expected)), 'recovered_bits': ''.join(map(str, actual))})
    n = len(test_files)
    conditions = ['float_clean'] + list(TEST_CONDITIONS)
    summary = [{'model': label, 'condition': c, 'n_images': n,
                **{k: float(np.mean([r[k] for r in records if r['condition'] == c]))
                   for k in ['bit_accuracy', 'exact_message', 'stamp_valid', 'valid_correct_verdict']}} for c in conditions]
    quality = {'model': label, 'png_stamped_psnr_db': psnr_sum / n, 'unstamped_png_accepted_count': unstamped_valid}
    return records, summary, quality

results = {}
for label, path in [('clean_start', INIT_FROM), ('jpeg_resize_best', RUN_DIR / 'best.pt')]:
    enc, dec, epoch = load_model(path)
    print(f'Testing {label} (epoch {epoch}) ...', flush=True)
    results[label] = run_test(enc, dec, f'{label}_epoch{epoch}')
    del enc, dec

all_records = [r for res in results.values() for r in res[0]]
all_summary = [r for res in results.values() for r in res[1]]
for name, rows_out in [('test_predictions.csv', all_records), ('test_summary.csv', all_summary)]:
    with open(RUN_DIR / name, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows_out[0]))
        writer.writeheader()
        writer.writerows(rows_out)
(RUN_DIR / 'quality_and_false_acceptance.json').write_text(json.dumps([res[2] for res in results.values()], indent=2))

clean_rows = {r['condition']: r for r in results['clean_start'][1]}
lines = [f'{"Condition":<24} {"Clean bits":>11} {"Trained bits":>13} {"Trained exact":>14} {"Trained valid+verdict":>22}']
for r in results['jpeg_resize_best'][1]:
    c = r['condition']
    lines.append(f'{c:<24} {clean_rows[c]["bit_accuracy"]:>11.2%} {r["bit_accuracy"]:>13.2%} '
                 f'{r["exact_message"]:>14.2%} {r["valid_correct_verdict"]:>22.2%}')
for label, res in results.items():
    q = res[2]
    lines.append(f'{q["model"]}: PNG PSNR {q["png_stamped_psnr_db"]:.2f} dB, unstamped accepted {q["unstamped_png_accepted_count"]}/{len(test_files)}')
log('\n'.join(['Test results (500 held-out photos):'] + lines))

### Example stamped image

The figure shows the original, the 8-bit stamped image and the absolute difference multiplied by 10. The cell also reports the recovered bits after real JPEG compression at quality 50 for this one example.

In [ ]:
enc, dec, epoch = load_model(RUN_DIR / 'best.pt')
path = test_files[0]
with Image.open(path) as image:
    original = test_transform(image.convert('RGB')).unsqueeze(0).to(device)
message = torch.tensor([fixed_payload(path.name)], dtype=torch.float32, device=device)
with torch.no_grad():
    stamped = pil_apply(((enc(original * 2 - 1, message) + 1) / 2).clamp(0, 1), [])
    after_jpeg = pil_apply(stamped, [('jpeg', 50)])
    recovered = read_bits(dec, after_jpeg).long()[0].tolist()
print('Expected          :', ''.join(map(str, message.long()[0].tolist())))
print('After real JPEG 50:', ''.join(map(str, recovered)), '| exact:', recovered == message.long()[0].tolist())
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for axis, img, title in zip(axes, [original[0], stamped[0], ((stamped[0] - original[0]).abs() * 10).clamp(0, 1)],
                            ['Original', 'Stamped (saved as PNG)', 'Difference ×10']):
    axis.imshow(img.permute(1, 2, 0).cpu()); axis.set_title(title); axis.axis('off')
plt.tight_layout()
fig.savefig(RUN_DIR / 'example_comparison.png', bbox_inches='tight')
plt.show()

### Save the results

Package the run folder, including logs, test results, figures and checkpoints, as a ZIP archive.

In [ ]:
bundle = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('Saved', bundle, f'({Path(bundle).stat().st_size / 1024**2:.1f} MB)')

## Summary

Bit accuracy on 500 test images, comparing the clean epoch-60 checkpoint with the selected fine-tuned checkpoint at epoch 97:

| Condition | Before fine-tuning | After fine-tuning |
|---|---:|---:|
| 8-bit pixels, PNG-equivalent | 99.76% | 99.98% |
| JPEG quality 90 | 50.49% | 98.24% |
| JPEG quality 70 | 50.48% | 92.08% |
| JPEG quality 50 | 50.50% | 86.24% |
| Resize 128→224→128 | 72.26% | 99.98% |
| Resize 128→64→128 | 50.84% | 82.33% |
| JPEG 70, then resize 128→224→128 | 50.42% | 86.58% |
| Centre crop to 112×112 | 62.89% | 73.59% |
| Mean stamped-image PSNR | 34.21 dB | 30.15 dB |

Fine-tuning improved JPEG recovery substantially. Bit accuracy exceeded 95% at quality 90, but not at qualities 70 or 50. Recovery after the 128→224→128 resize was also strong. More severe downscaling and cropping remained difficult.

Whole-message recovery was lower than bit accuracy: 57.2% of images had all 32 bits correct after JPEG 90, 8.8% after JPEG 70 and 1.0% after JPEG 50. The PNG-equivalent and 224-pixel round-trip conditions each recovered complete messages in 99.2% of images.

Mean PSNR fell by 4.06 dB relative to the clean epoch-60 model, showing the additional pixel distortion associated with this fine-tuning run. One of the 500 unstamped images passed the payload check. These results support improved robustness under the tested conditions, but do not establish reliable complete-message recovery under stronger JPEG compression.

## Sources

The model and losses build on [ando-khachatryan's PyTorch HiDDeN implementation](https://github.com/ando-khachatryan/HiDDeN/tree/556f76dd0602e4351ed4c19bd2ee87d50ef3c0de), credited with its MIT license in step 1. The U-Net-style encoder is described in step 2.

The mixed real/simulated JPEG training is adapted from [MBRS: Enhancing Robustness of DNN-based Watermarking by Mini-Batch of Real and Simulated JPEG Compression](https://arxiv.org/abs/2108.08211) by Jia, Fang and Zhang (ACM MM 2021), with [official code](https://github.com/jzyustc/MBRS). This experiment retains the HiDDeN loss weights and zigzag JPEG-Mask, adds resize training and uses actual Pillow JPEG in its real-compression branch.